In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Vocabulary = {"Attention" : 0, "is" : 1, "all" : 5, "you" : 2, "need" : 3, "!" : 6, "," : 4}

sentence = "Attention is all you need !"
tokens = sentence.split()

token_ids = []

for token in tokens:
    token_ids.append(Vocabulary[token])

#Here we define the target ids by shifting the token_ids by one element to the left.  
target_ids = token_ids[1:]
#input delets the last cell. For our example being "!". 
input_ids = token_ids[:-1]

#Then we convert this ordinary list into torch tensor.
input_ids = torch.tensor(input_ids, dtype = torch.long)
target_ids = torch. tensor(target_ids, dtype = torch.long)

#Here we deine the variables, that we will later use to define matrices. 
#N = len(token_ids) size of tokens, but it's not needed to define inside init function. Later we will define it in forward functino using shape[0]. I just mention what it was so I won't forget to define in forward function. 
V = len(Vocabulary) #vocab size
d_model = V
d_k = 4
d_v = 5


loss_history = []

#A class describes objects that can store information. 
class ToyAttentionModel(nn.Module):

    #Here we define a special function attached to a class. 
    def __init__(self, vocab_size, d_model, d_k, d_v): 
        super().__init__()

        #definint E
        self.embedding = nn.Embedding(vocab_size, d_model)
        #defining W_Q
        self.query = nn.Linear(d_model, d_k, bias = False)
        #W_K
        self.key = nn.Linear(d_model, d_k, bias = False)
        #W_V
        self.value = nn.Linear(d_model, d_v, bias = False)
        #W_vocab
        self.output = nn.Linear(d_v, vocab_size, bias = False)
    # Here I define forward function. 
    def forward(self, input_ids):
        X = self.embedding(input_ids)
        Q = self.query(X)
        K = self.key(X)
        V_values = self.value(X) #X @ W_V
        d_k = Q.shape[1]
        S = (Q @ K.T)/(d_k **(0.5)) #score calculation
        N = input_ids.shape[0] # number of tokens
        M = torch.full((N,N), float("-inf"), dtype = X.dtype, device = X.device)#caueal Masking being defined to make current approximation independent of future tokens.
        M = torch.triu(M, diagonal = 1)
        A = torch.softmax(S + M, dim = 1)
        O = A @ V_values
        Z = self.output(O) #getting back to the definition of self.output, we defined as Linear(d_v, vocab_size), so it takes Z = O @ W_vocab, which we called this W_vocab as self.output.  
    
        #return X, Q, K, V_values, A, O, Z, well I only need logits. Just wanted to check for dimension and that everything works fine, which did. So, now I only have forward to return logits(Z).
        return Z

#here we first instantiate the model by calling it, and naming it model. 
model = ToyAttentionModel(
    vocab_size=V,
    d_model=d_model,
    d_k=d_k,
    d_v=d_v
)
for name, parameter in model.named_parameters():
    print(name, parameter.shape)





embedding.weight torch.Size([7, 7])
query.weight torch.Size([4, 7])
key.weight torch.Size([4, 7])
value.weight torch.Size([5, 7])
output.weight torch.Size([7, 5])


In [ ]:
# Run the completed forward pass and verify that its logits match the targets.
logits = model(input_ids)

print("input_ids shape:", input_ids.shape)
print("target_ids shape:", target_ids.shape)
print("logits shape:", logits.shape)
